# 04 · Binary Search

Binary search finds a value in a **sorted** array by repeatedly cutting the search range in half.

- Look at the middle element `nums[mid]`.
- If it equals the target → done.
- If it is smaller than the target → the answer must be on the **right** (`low = mid + 1`).
- If it is larger → the answer must be on the **left** (`high = mid - 1`).

**Time:** O(log n) · **Space:** O(1) iterative, O(log n) recursive

**Topics covered**
1. Classic binary search (iterative + recursive)
2. Lower bound & upper bound
3. Search insert position
4. Floor and ceil
5. First & last occurrence, count occurrences
6. Search in a rotated sorted array (distinct values and with duplicates)
7. Minimum in a rotated sorted array

## 1. Classic binary search

The loop condition is `low <= high` (both ends are still valid candidates). `mid` must be **recomputed inside the loop** every iteration.

> 🐞 **Fix:** the earlier version calculated `mid` once *before* the loop and printed `-1` even after finding the target. Here we `return` as soon as we find it.

In [1]:
def binary_search(nums, target):
    low, high = 0, len(nums) - 1
    while low <= high:
        mid = (low + high) // 2          # recompute every time
        if nums[mid] == target:
            return mid
        elif nums[mid] < target:
            low = mid + 1                # discard the left half
        else:
            high = mid - 1               # discard the right half
    return -1                            # range became empty → not found

nums = [2, 4, 6, 7, 9, 11, 18, 19]
print(binary_search(nums, 18))           # 6
print(binary_search(nums, 13))           # -1

6
-1


### Recursive version
Same logic, but the "loop" is a function that calls itself on the smaller range. **Base case:** `low > high` → not found.

In [2]:
def binary_search_rec(nums, low, high, target):
    if low > high:                       # base case
        return -1
    mid = (low + high) // 2
    if nums[mid] == target:
        return mid
    elif nums[mid] < target:
        return binary_search_rec(nums, mid + 1, high, target)
    else:
        return binary_search_rec(nums, low, mid - 1, target)

nums = [2, 4, 6, 7, 9, 11, 18, 19]
print(binary_search_rec(nums, 0, len(nums) - 1, 18))   # 6

6


## 2. Lower bound & upper bound

These are the building blocks for most "find the boundary" problems.

| Name | Meaning |
|---|---|
| **Lower bound** | smallest index `i` with `nums[i] >= target` |
| **Upper bound** | smallest index `i` with `nums[i] > target` |

If no such index exists, both return `n` (one past the end), which is exactly where the target *would be inserted*.

The pattern: when `nums[mid]` satisfies the condition, **save `mid` as a possible answer**, then keep searching on the left for something even smaller.

In [3]:
def lower_bound(nums, target):
    n = len(nums)
    ans = n                              # default: "not found" = n
    low, high = 0, n - 1
    while low <= high:
        mid = (low + high) // 2
        if nums[mid] >= target:
            ans = mid                    # candidate; try to find a smaller index
            high = mid - 1
        else:
            low = mid + 1
    return ans

def upper_bound(nums, target):
    n = len(nums)
    ans = n
    low, high = 0, n - 1
    while low <= high:
        mid = (low + high) // 2
        if nums[mid] > target:           # strictly greater
            ans = mid
            high = mid - 1
        else:
            low = mid + 1
    return ans

nums = [1, 1, 1, 1, 1, 2, 3, 3, 5, 6, 7, 7, 7, 7, 9, 12, 12, 13]
print("lower_bound(1) =", lower_bound(nums, 1))   # 0
print("upper_bound(1) =", upper_bound(nums, 1))   # 5
print("lower_bound(4) =", lower_bound(nums, 4))   # 8  (index where 4 would be inserted)
print("upper_bound(99) =", upper_bound(nums, 99)) # 18 = len(nums)

lower_bound(1) = 0
upper_bound(1) = 5
lower_bound(4) = 8
upper_bound(99) = 18


## 3. Search insert position

Return the index of `target` if present, otherwise the index where it should be inserted to keep the array sorted. **That is exactly the lower bound.**

The earlier linear-scan attempt printed multiple answers and misbehaved at index 0 (`nums[i-1]` wraps to the last element in Python). Using `lower_bound` avoids all those special cases.

In [4]:
def search_insert(nums, target):
    return lower_bound(nums, target)

nums = [1, 3, 4, 5, 8, 9, 14, 15, 19, 20, 21]
print(search_insert(nums, 3))     # 1  (found)
print(search_insert(nums, 6))     # 4  (would go between 5 and 8)
print(search_insert(nums, 0))     # 0  (before everything)
print(search_insert(nums, 100))   # 11 (after everything)

1
4
0
11


## 4. Floor and ceil

- **Floor** = largest value `<= target`
- **Ceil** = smallest value `>= target`

While searching: if `nums[mid] > target`, it's a ceil candidate (go left); if `nums[mid] < target`, it's a floor candidate (go right). An exact match is both. Missing values stay `-1`.

In [5]:
def floor_ceil(nums, target):
    floor = ceil = -1
    low, high = 0, len(nums) - 1
    while low <= high:
        mid = (low + high) // 2
        if nums[mid] == target:
            return [nums[mid], nums[mid]]
        elif nums[mid] > target:
            ceil = nums[mid]
            high = mid - 1
        else:
            floor = nums[mid]
            low = mid + 1
    return [floor, ceil]

nums = [1, 3, 4, 5, 8, 9, 14, 15, 19, 20, 21]
print(floor_ceil(nums, 4))     # [4, 4]
print(floor_ceil(nums, 6))     # [5, 8]
print(floor_ceil(nums, 0))     # [-1, 1]

[4, 4]
[5, 8]
[-1, 1]


## 5. First & last occurrence, and count

- **First occurrence** = `lower_bound(target)` (if `nums[lb] == target`)
- **Last occurrence** = `upper_bound(target) - 1`
- **Count** = `upper_bound − lower_bound`  → O(log n) instead of scanning in O(n)

In [6]:
class Solution:
    def search_range(self, nums, target):
        lb = lower_bound(nums, target)
        if lb == len(nums) or nums[lb] != target:    # target isn't in the array
            return [-1, -1]
        ub = upper_bound(nums, target)
        return [lb, ub - 1]

nums = [1, 1, 3, 4, 4, 4, 4, 5, 7, 8, 8, 8, 9, 9, 9, 10]
print("Range of 9 :", Solution().search_range(nums, 9))    # [12, 14]
print("Range of 6 :", Solution().search_range(nums, 6))    # [-1, -1]

# Count occurrences
def count_occurrences(nums, target):
    return upper_bound(nums, target) - lower_bound(nums, target)

print("Count of 3 :", count_occurrences(nums, 3))           # 1
print("Count of 4 :", count_occurrences(nums, 4))           # 4

Range of 9 : [12, 14]
Range of 6 : [-1, -1]
Count of 3 : 1
Count of 4 : 4


### Linear search for comparison (O(n))

In [7]:
nums = [1, 1, 3, 4, 4, 4, 4, 5, 7, 8, 8, 8, 9, 9, 9, 10]

def first_index_linear(nums, target):
    for i in range(len(nums)):
        if nums[i] == target:
            return i
    return -1

def count_linear(nums, target):
    count = 0
    for x in nums:
        if x == target:
            count += 1
    return count

print(first_index_linear(nums, 10), count_linear(nums, 3))   # 15 1

15 1


## 6. Search in a rotated sorted array

A sorted array was rotated at some pivot, e.g. `[7, 8, 9, 10, 1, 2, 3, 5, 6]`.
It is no longer fully sorted, but **at least one half is always sorted** around `mid`.

**Logic at each step**
1. If `nums[mid] == target` → return.
2. Find the sorted half:
   - `nums[mid] <= nums[high]` → the **right** half is sorted.
   - otherwise the **left** half is sorted.
3. Check if the target lies inside the sorted half. If yes, search there; if no, search the other half.

**Time:** O(log n) for distinct values.

In [8]:
def search_rotated(nums, target):
    low, high = 0, len(nums) - 1
    while low <= high:
        mid = (low + high) // 2
        if nums[mid] == target:
            return mid
        if nums[mid] <= nums[high]:              # right half [mid..high] is sorted
            if nums[mid] <= target <= nums[high]:
                low = mid + 1                    # target is in the sorted right half
            else:
                high = mid - 1
        else:                                    # left half [low..mid] is sorted
            if nums[low] <= target <= nums[mid]:
                high = mid - 1                   # target is in the sorted left half
            else:
                low = mid + 1
    return -1

nums = [7, 8, 9, 10, 1, 2, 3, 5, 6]
print(search_rotated(nums, 8))     # 1
print(search_rotated(nums, 3))     # 6
print(search_rotated(nums, 4))     # -1

1
6
-1


### With duplicates

When `nums[low] == nums[mid] == nums[high]` we **can't tell which half is sorted**. Example: `[1, 1, 1, 0, 1]`. The only safe move is to shrink both ends by one (`low += 1`, `high -= 1`). In the worst case (all equal) this degrades to O(n).

First, see why the version above isn't enough:

In [9]:
print(search_rotated([1, 1, 1, 0, 1], 0))   # -1  ❌ wrong! 0 is in the array

-1


In [10]:
def search_rotated_duplicates(nums, target):
    low, high = 0, len(nums) - 1
    while low <= high:
        mid = (low + high) // 2
        if nums[mid] == target:
            return True
        if nums[low] == nums[mid] == nums[high]:     # can't decide → trim both ends
            low += 1
            high -= 1
        elif nums[mid] <= nums[high]:                # right half sorted
            if nums[mid] <= target <= nums[high]:
                low = mid + 1
            else:
                high = mid - 1
        else:                                        # left half sorted
            if nums[low] <= target <= nums[mid]:
                high = mid - 1
            else:
                low = mid + 1
    return False

print(search_rotated_duplicates([1, 1, 1, 0, 1], 0))                        # True ✅
print(search_rotated_duplicates([7, 7, 7, 7, 7, 7, 7, 1, 7, 7, 7, 7], 6))   # False
print(search_rotated_duplicates([12, 12, 13, 14, 1, 2, 2, 3, 3, 3, 4, 9, 9, 9, 10, 10, 11, 11], 9))  # True

True
False
True


## 7. Minimum in a rotated sorted array

**Brute force:** scan everything → O(n). (Start `min` at `float("inf")`, and don't name a variable `min`, otherwise it hides the built-in `min()` function.)

**Binary search:** in each step one half is sorted, so its **smallest value is at its left end**. Record that candidate, then discard that half and search the other.

**Time:** O(log n)

In [11]:
# Brute force
def min_brute(nums):
    smallest = float("inf")
    for x in nums:
        smallest = min(smallest, x)
    return smallest

# Binary search
def min_rotated(nums):
    low, high = 0, len(nums) - 1
    ans = float("inf")
    while low <= high:
        mid = (low + high) // 2
        if nums[mid] <= nums[high]:              # right half sorted → its min is nums[mid]
            ans = min(ans, nums[mid])
            high = mid - 1                       # the pivot might still be on the left
        else:                                    # left half sorted → its min is nums[low]
            ans = min(ans, nums[low])
            low = mid + 1
    return ans

nums = [7, 8, 9, 1, 2, 3, 4]
print(min_brute(nums), min_rotated(nums))        # 1 1

1 1


---
### ✅ Key takeaways
- Binary search needs a **sorted** (or at least *monotonic*) search space.
- **Lower bound / upper bound** solve insert position, first/last occurrence and counting.
- **Rotated array:** one half is always sorted — find it, test whether the target lies in it, then discard a half.
- With **duplicates**, handle `nums[low] == nums[mid] == nums[high]` by shrinking both ends.